# Quantum Phase Estimation

Quantum Phase Estimation (QPE) estimates the eigenvalue $e^{i\theta}$ of a unitary operator $U$ given an eigenstate $|\psi\rangle$ such that $U|\psi\rangle = e^{i\theta}|\psi\rangle$.

The circuit uses $t$ count qubits to estimate $\theta$ to $t$ bits of precision:
1. Initialize count qubits in $|+\rangle$ superposition
2. Apply controlled-$U^{2^k}$ from count qubit $k$ to the target
3. Apply inverse QFT to the count register
4. Measure the count register

In [ ]:
import cudaq
import numpy as np


@cudaq.kernel
def qpe_rz(theta: float):
    """Phase estimation for RZ(theta) using 2 count qubits."""
    qubits = cudaq.qvector(4)
    x(qubits[3])
    h(qubits[0])
    h(qubits[1])
    crz(qubits[0], qubits[3], theta)
    crz(qubits[1], qubits[3], 2 * theta)
    swap(qubits[0], qubits[1])
    h(qubits[0])
    crz(qubits[0], qubits[1], -np.pi / 2)
    h(qubits[1])

In [ ]:
test_angles = [np.pi / 2, np.pi, 3 * np.pi / 2]
for theta in test_angles:
    print(f"--- RZ({np.degrees(theta):.1f} deg) ---")
    result = cudaq.sample(qpe_rz, theta, shots_count=1000)
    for bitstring, count in result.items():
        count_val = int(bitstring[0:2], 2)
        phase_est = count_val * 2 * np.pi / 4
        print(f"  measured |{bitstring}>: {count} "
              f"(phase ~ {phase_est:.4f} rad = {np.degrees(phase_est):.1f} deg)")
    print()

QPE is the core subroutine of Shor's factoring algorithm, quantum simulation (eigenvalue estimation), and the HHL linear systems algorithm.